In [1]:
# for a given neuron, compute the IG score for each token

from LKN.utils import get_llm_block, get_model

model_name = "meta-llama/Meta-Llama-3-8B-Instruct"
llm, tokenizer = get_model(model_name)

/home/bumjin/anaconda3/lib/python3.11/site-packages/transformers/utils/hub.py:110: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(
`torch_dtype` is deprecated! Use `dtype` instead!
/home/bumjin/anaconda3/lib/python3.11/site-packages/torchvision/io/image.py:13: UserWarning: Failed to load image Python extension: '/home/bumjin/anaconda3/lib/python3.11/site-packages/torchvision/image.so: undefined symbol: _ZN3c1017RegisterOperatorsD1Ev'If you don't plan on using image functionality from `torchvision.io`, you can ignore this warning. Otherwise, there might be something wrong with your environment. Did you have `libjpeg` or `libpng` installed before building `torchvision` from source?
  warn(
2025-11-26 07:40:45.082042: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation 

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

In [2]:
import torch 
from LKN.utils import  get_llm_block

class BackpropHook:
        def __init__(self):
            self.hidden_states = None
            self.probe_positions = None
            
        def __call__(self, module, input, output):
            # Store hidden states on CPU immediately and convert to float16
            self.hidden_states = []
            for i  in range(self.probe_positions.shape[0]):
                probe_position = self.probe_positions[i]
                self.hidden_states.append(input[0][i, probe_position, :])
            self.hidden_states = torch.stack(self.hidden_states)
            
        def clear(self):
            self.hidden_states = None

blocks = get_llm_block(llm, model_name)
input = "hello world!, I am a student, How are you?"

tokens = tokenizer.encode(input, return_tensors='pt').to(llm.device)  
embed_layer = llm.get_input_embeddings()           # nn.Embedding
embeds = embed_layer(tokens)                       # [1, T, d]
embeds.retain_grad()                               # 이 텐서에 grad 저장

hook = BackpropHook()
hook.probe_positions = torch.tensor([[ -2, -1]])  # batch=1 기준

target_layer = 11
target_neuron = 10

layer_module = blocks[target_layer]
layer_module.register_forward_hook(hook)

llm.zero_grad()
# ② inputs_embeds로 모델에 넣기 (ids 대신)
output = llm(inputs_embeds=embeds)

# ③ 타겟 뉴런에서 backward
hook.hidden_states[:, :, target_neuron].sum().backward()
input_grad = embeds.grad * embeds
batch_attribution = input_grad.abs().mean(dim=-1)

In [3]:
batch_index = 0 
for token, attr in zip(tokenizer.convert_ids_to_tokens(tokens[batch_index]), 
                       batch_attribution[batch_index]):
    print(f"{token} : {attr:.4f}")

<|begin_of_text|> : 0.0000
hello : 0.0001
Ġworld : 0.0001
!, : 0.0002
ĠI : 0.0001
Ġam : 0.0000
Ġa : 0.0000
Ġstudent : 0.0001
, : 0.0000
ĠHow : 0.0001
Ġare : 0.0001
Ġyou : 0.0001
? : 0.0001
